# Данные: состав, область изображения и разметка

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Цель: проверить исходные изображения до моделей. Девять файлов — пилот, не репрезентативный benchmark.
Все метки изначально пустые. В `data/labels.csv` человек записывает точную строку и ставит
`label_status=verified`; неоднозначные изображения отмечаются `ambiguous`.
Ответы моделей не подставляются в эталон автоматически.

In [ ]:
from captcha_lab.data import import_images
# При другом расположении файлов измените путь. После импорта исходная папка не нужна.
import os
from dotenv import load_dotenv
load_dotenv(ROOT/'.env.local',override=False)
SOURCE = Path(os.getenv('CAPTCHA_SOURCE_DIR',str(ROOT/'data/incoming')))
if SOURCE.is_dir(): print('Импортировано:', import_images(SOURCE))
manifest = image_manifest()
assert not manifest.empty, 'Скопируйте изображения в data/local или укажите SOURCE'
assert manifest.error.eq('').all(), 'Есть поврежденные изображения'
display(manifest.drop(columns=['path']))
print('Дубликаты SHA-256:', int(manifest.sha256.duplicated().sum()))
labels = ensure_labels(manifest)
display(labels)

Область ниже выбрана вручную для предоставленного макета экрана. Она не является универсальным
детектором капчи. Перед прогоном проверьте, что символы целиком внутри каждой вырезки.
RGB и grayscale сравниваются без удаления линий: агрессивная обработка может удалить части символов.

In [ ]:
import matplotlib.pyplot as plt
from captcha_lab.data import DEFAULT_ROI
fig, axes = plt.subplots(int(np.ceil(len(manifest)/3)),3,figsize=(14,3*np.ceil(len(manifest)/3)),squeeze=False)
for ax in axes.flat: ax.axis('off')
for ax, row in zip(axes.flat, manifest.itertuples()):
    ax.imshow(crop_image(row.path)); ax.set_title(row.image_id)
fig.suptitle('Ручная область изображения; исходные 9 примеров из ТЗ')
plt.tight_layout(); plt.show()
print('ROI:', DEFAULT_ROI)

In [ ]:
out = manifest.copy()
out['path'] = out.path.map(lambda p: str(Path(p).relative_to(ROOT)))
(ROOT/'results').mkdir(exist_ok=True)
out.to_csv(ROOT/'results/input_manifest.csv',index=False)
print('Проверенных меток:', int(labels.label_status.eq('verified').sum()))